# 02. 따릉이 × 지하철 시각화 (Parquet 읽기 전용)

`01_전처리_따릉이_지하철.ipynb`가 만든 Parquet을 **읽기만** 해서 그래프·상관계수를 보여줍니다.
이 노트북은 파일을 만들거나 바꾸지 않습니다 (Parquet 저장 코드 없음).

## 목차: 위에서부터 이 순서대로 실행
| 순서 | 절 | 보여주는 것 (한 줄 요약) | 읽는 파일 (`data/parquet_bike_subway/`) |
|---|---|---|---|
| 0 | 준비 | 라이브러리·한글 폰트 · 경로 · 데이터 불러오기 · 그래프 함수 | 전체 |
| 1-1 | 기본 EDA | 시간대별 하루 평균 대여·반납 | `mart_station_month_hour/` |
| 1-2 | 〃 | 평일·주말·공휴일 시간대 패턴 + 평일 피크 상위 5 | 〃 , `calendar_day` |
| 1-3·1-4 | 〃 | 대여소별 월평균 대여 vs 반납 산점도 + net_flow 상·하위 10 | 〃 |
| 1-5·1-6 | 〃 | 출근시간(07~09시) 평일 순유입·순유출 상위 15곳 | 〃 , `bike_station_both` |
| 1-7 | 〃 | 집계 검산 | 〃 |
| 2-1 | 지하철 연관 | 시간대별 상관 (역 승차↔대여, 역 하차↔반납) | `station_subway_link`, `subway_ride_month_hour` |
| 2-2 | 〃 | 대여소 단위 Spearman 4가지 조합 | 〃 |
| 2-3 | 〃 | 역 거리 구간별 평균 대여·반납 | 〃 |
| 2-4 | 〃 | 같은 구 안에서 역세권 vs 비역세권 | 〃 , `bike_station_both` |
| 2-5 | 〃 | 좌표 없는 대여소의 영향 | `feature_station_subway` |
| 2-6 | 〃 | 거리 구간 재계산 (좌표 없는 대여소 제외) | 〃 |
| 2-7 | 〃 (추가) | 대여소 변수 상관 행렬 히트맵 | 〃 |
| 2-8 | 결론 | 이번 기간 자동 요약 + 기존 결론(참고) | - |
| 3 | 추가 시각화 | 대여소 분포 · 역까지 거리 분포 · 지하철 승하차 추이 · 일별 추정 확인 | `bike_station_nearest`, `subway_*` |

## 실행 전 확인
1. `01_전처리_따릉이_지하철.ipynb`를 **끝까지** 실행해 `data/parquet_bike_subway/`가 있어야 합니다.
2. 분석 기간은 01 노트북이 남긴 `_logs/manifest.json`에서 자동으로 읽습니다 (여기서 바꾸지 않음).
3. 경로는 `0-2 설정`에서만 바꿉니다 (01 노트북과 같은 `PROJECT_DIR`).

# 0. 준비
## 0-1. 라이브러리 · 한글 폰트
**왜:** 그래프에 한글이 깨지지 않게 Colab은 나눔고딕을 설치하고, 로컬은 설치된 한글 폰트(맑은 고딕 등)를 찾습니다.

In [ ]:
import importlib, subprocess, sys

def ensure_package(name, pip_name=None):
    # 없을 때만 설치
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or name])

for _n in ["pandas", "numpy", "pyarrow", "matplotlib"]:
    ensure_package(_n)

import os, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator
from IPython.display import display

try:                                   # Colab이면 구글 드라이브 연결
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# 한글 폰트: Colab은 나눔고딕 설치, 로컬은 설치된 한글 폰트 사용
if IN_COLAB and not any("NanumGothic" in f.name for f in fm.fontManager.ttflist):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
    for _p in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf"]:
        if os.path.exists(_p):
            fm.fontManager.addfont(_p)
KOREAN_FONTS = ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR", "Noto Sans CJK JP", "Noto Sans KR"]
_installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((f for f in KOREAN_FONTS if f in _installed), None)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", message="Glyph .* missing from font")
print("pandas", pd.__version__, "/ matplotlib", matplotlib.__version__, "/ 한글 폰트:", FONT or "없음 (글자가 네모로 보일 수 있음)")

## 0-2. 설정 ⭐ (경로만 바꾸면 됩니다)
**왜:** 01 노트북과 같은 폴더를 읽고, 분석 기간은 01이 남긴 기록(`manifest.json`)을 그대로 써서 기간이 섞이지 않게 합니다.

| 설정 | 뜻 |
|---|---|
| `PROJECT_DIR` | 프로젝트 폴더 (Colab: 드라이브 / 로컬: 환경변수 `BIKE_PROJECT_DIR`, 없으면 현재 폴더) |
| `OUT_DIR` | 01 노트북 결과 폴더 (읽기만 함) |
| `START_DATE`, `END_DATE` | 분석 기간 (manifest에서 읽음) |
| `NEAR_RADIUS_M`, `MAX_LINK_RADIUS_M` | 역세권 300m / 연결 최대 500m (01과 같은 값) |

In [ ]:
if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/final_project")
else:
    PROJECT_DIR = Path(os.environ.get("BIKE_PROJECT_DIR", ".")).resolve()
OUT_DIR = PROJECT_DIR / "data" / "parquet_bike_subway"          # 01 노트북 결과 (읽기 전용)

manifest = OUT_DIR / "_logs" / "manifest.json"
assert manifest.exists(), f"{manifest} 가 없습니다 → 01_전처리_따릉이_지하철.ipynb 를 먼저 끝까지 실행하세요."
settings = json.loads(manifest.read_text(encoding="utf-8"))
START_DATE, END_DATE = settings["start_date"], settings["end_date"]
PERIOD_LABEL = f"{START_DATE} ~ {END_DATE}"

NEAR_RADIUS_M, MAX_LINK_RADIUS_M = 300, 500                    # 01 노트북과 같은 값
SEOUL_LAT, SEOUL_LON = (37.0, 38.2), (126.0, 128.0)

REQUIRED = ["calendar_day.parquet", "bike_station_both.parquet", "station_subway_link.parquet",
            "bike_station_nearest.parquet", "subway_station_phys.parquet", "subway_ride_month_hour.parquet",
            "feature_station_subway.parquet", "mart_station_month_hour"]
missing = [f for f in REQUIRED if not (OUT_DIR / f).exists()]
assert not missing, f"01 노트북 결과가 없습니다: {missing}"
print("결과 폴더:", OUT_DIR)
print("분석 기간:", PERIOD_LABEL)

## 0-3. 데이터 불러오기
**왜:** 이후 모든 그래프가 같은 표를 쓰도록 한 번에 읽어 둡니다.

| 변수 | 파일 | 한 행 | 주요 컬럼 - 뜻 |
|---|---|---|---|
| `cal` | `calendar_day` | 날짜 | `cal_date` 날짜, `ym` 연월, `day_type` weekday/weekend/holiday |
| `df_A` | `mart_station_month_hour/` | 대여소 × 연월 × 요일구분 × 시간 | `station_id` 대여소, `rent_count` 대여, `return_count` 반납, `n_days` 그 달 해당 요일구분 일수 |
| `df_both` | `bike_station_both` | 대여소 | `rent_id` 대여소 ID, `rent_nm` 이름, `sta_loc` 자치구, `hold_num` 거치대 수 |
| `df_link` | `station_subway_link` | 대여소 × 500m 안 역 | `phys_key` 물리역, `distance_m` 거리, `radius_band` 거리 구간, `is_nearest` 가장 가까운 역 |
| `df_nearest` | `bike_station_nearest` | 대여소 | `nearest_any_m` 거리 제한 없이 가장 가까운 역까지 거리 |
| `df_subway_station` | `subway_station_phys` | 물리역 | `subway_name` 역명, `lines` 호선 |
| `df_subway_ride` | `subway_ride_month_hour` | 물리역 × 연월 × 시간 | `get_on` 승차, `get_off` 하차 (월 합계) |
| `feature_station_subway` | 같은 이름 | 대여소 | `has_coord` 좌표 정상, `nearest_distance_m` 500m 안 가장 가까운 역 거리 |

In [ ]:
def read_parquet_dir(folder, columns=None):
    # 폴더 안 Parquet 조각을 합쳐 읽기
    files = sorted(Path(folder).glob("*.parquet"))
    return pd.concat([pd.read_parquet(f, columns=columns) for f in files], ignore_index=True) if files else pd.DataFrame()


def smatch(s, pattern):
    # 정규식 일치 여부 (결측은 False)
    return s.astype("string").str.match(pattern).fillna(False).astype(bool)


def valid_coord(lat, lon):
    return lat.between(*SEOUL_LAT) & lon.between(*SEOUL_LON)


cal                    = pd.read_parquet(OUT_DIR / "calendar_day.parquet")
df_A                   = read_parquet_dir(OUT_DIR / "mart_station_month_hour")
df_both                = pd.read_parquet(OUT_DIR / "bike_station_both.parquet")
df_link                = pd.read_parquet(OUT_DIR / "station_subway_link.parquet")
df_nearest             = pd.read_parquet(OUT_DIR / "bike_station_nearest.parquet")
df_subway_station      = pd.read_parquet(OUT_DIR / "subway_station_phys.parquet")
df_subway_ride         = pd.read_parquet(OUT_DIR / "subway_ride_month_hour.parquet")
feature_station_subway = pd.read_parquet(OUT_DIR / "feature_station_subway.parquet")
_full = OUT_DIR / "subway_ride_day_hour_full.parquet"                 # 6절 실측이 있을 때만 존재
df_subway_full = pd.read_parquet(_full) if _full.exists() else pd.DataFrame()

# 기간 파생값 (평균을 낼 때 나누는 일수)
DAYS_IN_PERIOD = len(cal)                                   # 전체 일수
DAYS_BY_TYPE   = cal["day_type"].value_counts()             # 요일구분별 일수
days_ym        = cal.groupby("ym").size().rename("days_in_month")   # 연월별 일수
N_MONTHS       = cal["ym"].nunique()
assert df_A["ym"].nunique() == N_MONTHS, "집계표 월 수가 기간과 다릅니다 → 01 노트북 4절을 끝까지 실행하세요."

print(f"기간 {DAYS_IN_PERIOD}일 ({N_MONTHS}개월) / 집계표 {len(df_A):,}행 / 대여소 {df_A['station_id'].nunique():,}곳")
print(f"연결 쌍 {len(df_link):,} / 물리역 {len(df_subway_station):,} / 지하철 월별 {df_subway_ride['ym'].nunique()}개월"
      f" / 일별 추정 표 {'있음' if len(df_subway_full) else '없음'}")

## 0-4. 그래프 스타일 · 그리기 함수 (기존 3_eda 유지)
**왜:** 모든 그래프를 같은 색·제목 규칙으로 그려 비교하기 쉽게 합니다.

- 색: 파랑 = 대여·승차·순유입 / 주황 = 반납·하차 / 청록 = 공휴일 / 빨강 = 순유출
- 부제목에 기간(`PERIOD_LABEL`)을 적고, 피크 같은 핵심 지점만 직접 라벨

| 함수 | 그래프 |
|---|---|
| `plot_hourly_all` | 1-1 시간대별 하루 평균 대여·반납 |
| `plot_hourly_daytype` | 1-2 요일구분별 시간대 패턴 |
| `plot_station_scatter` | 1-3·1-4 대여소별 월평균 대여 vs 반납 |
| `plot_commute_netflow` | 1-5·1-6 출근시간 순유입·순유출 상위 |
| `plot_bar_h` | 3절 가로 막대 |

In [ ]:
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, RED  = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.unicode_minus": False,
    **({"font.family": FONT} if FONT else {}),
})
_thousands = FuncFormatter(lambda v, _: f"{v:,.0f}")


def _title(ax, text, sub=None):
    # 제목은 왼쪽 정렬, 부제목은 축 위 5pt 고정 간격
    ax.set_title(text, loc="left", color=INK, fontsize=13, fontweight="bold", pad=26 if sub else 10)
    if sub:
        ax.annotate(sub, (0, 1), xycoords="axes fraction", xytext=(0, 5), textcoords="offset points",
                    color=INK2, fontsize=9, va="bottom")


def plot_hourly_all(df):
    # [1-1] 시간대별 하루 평균 대여·반납 (선 2개, 피크만 라벨)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for col, color, name, dy in [("avg_rent_per_day", BLUE, "대여", 10), ("avg_return_per_day", ORANGE, "반납", -16)]:
        ax.plot(df["hour"], df[col], color=color, lw=2, label=name)
        i = df[col].idxmax()
        ax.scatter([df.loc[i, "hour"]], [df.loc[i, col]], s=45, color=color, edgecolor=SURFACE, linewidth=2, zorder=3)
        ax.annotate(f"{name} 피크 {int(df.loc[i, 'hour'])}시 · {df.loc[i, col]:,.0f}건",
                    (df.loc[i, "hour"], df.loc[i, col]), xytext=(10, dy), textcoords="offset points",
                    color=INK2, fontsize=9)
    ax.set_xticks(range(0, 24, 2)); ax.set_xlim(-0.5, 23.5); ax.set_ylim(bottom=0)
    ax.set_xlabel("시간(시)"); ax.set_ylabel("하루 평균 건수 (전체 대여소 합)")
    ax.yaxis.set_major_formatter(_thousands)
    ax.legend(frameon=False, loc="upper left", labelcolor=INK2)
    _title(ax, "시간대별 하루 평균 대여·반납 건수", f"{PERIOD_LABEL} · 모든 요일 · 전체 대여소 합계")
    fig.tight_layout(); plt.show()


def plot_hourly_daytype(df):
    # [1-2] 평일/주말/공휴일 시간대 패턴: 대여|반납 두 칸, 같은 y축
    kinds = [("weekday", "평일", BLUE, 4), ("weekend", "주말", ORANGE, 16), ("holiday", "공휴일", AQUA, 28)]   # 마지막 값: 라벨 겹침 방지 간격
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
    for ax, col, ttl in [(axes[0], "avg_rent_per_day", "대여"), (axes[1], "avg_return_per_day", "반납")]:
        for key, name, color, dy in kinds:
            d = df[df["day_type"] == key].sort_values("hour")
            if d.empty:
                continue
            ax.plot(d["hour"], d[col], color=color, lw=2, label=f"{name} ({int(d['n_days'].iloc[0]):,}일)")
            j = d[col].idxmax()
            ax.annotate(name, (d.loc[j, "hour"], d.loc[j, col]), xytext=(8, dy - 16), textcoords="offset points",
                        color=INK2, fontsize=9)
        ax.set_xticks(range(0, 24, 3)); ax.set_xlim(-0.5, 23.5); ax.set_ylim(bottom=0)
        ax.set_xlabel("시간(시)"); ax.yaxis.set_major_formatter(_thousands)
        ax.set_title(ttl, loc="left", color=INK, fontsize=11, fontweight="bold")
    axes[0].set_ylabel("하루 평균 건수 (전체 대여소 합)")
    axes[0].legend(frameon=False, loc="upper left", labelcolor=INK2)
    fig.suptitle("요일구분별 시간대 패턴: 평일 출근시간 피크가 주말·공휴일에도 있는가", x=0.01, ha="left",
                 color=INK, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()


def plot_station_scatter(df):
    # [1-3,1-4] 대여소별 월평균 대여 vs 반납. 대각선 위 = 반납 우세(순유입), 아래 = 대여 우세(순유출)
    fig, ax = plt.subplots(figsize=(6.6, 6.2))
    ax.scatter(df["rent_per_month"], df["return_per_month"], s=16, color=BLUE, alpha=0.4, edgecolor="none")
    m = float(max(df["rent_per_month"].max(), df["return_per_month"].max()))
    ax.plot([0, m], [0, m], color=INK2, lw=1, ls="--")
    ax.text(m * 0.98, m * 0.93, "대여 = 반납", ha="right", color=INK2, fontsize=9)
    ax.set_xlim(0, m * 1.02); ax.set_ylim(0, m * 1.02); ax.set_aspect("equal")
    ax.set_xlabel("월평균 대여 건수"); ax.set_ylabel("월평균 반납 건수")
    ax.xaxis.set_major_formatter(_thousands); ax.yaxis.set_major_formatter(_thousands)
    _title(ax, "대여소별 월평균 대여 vs 반납", f"점 하나 = 대여소 1곳 ({len(df):,}곳) · {PERIOD_LABEL}")
    fig.tight_layout(); plt.show()


def plot_commute_netflow(df, top_n=15, hours_label="07~09시"):
    # [1-5,1-6] 출근시간대 평일 순유입/순유출 상위. 파랑 = 순유입, 빨강 = 순유출, 0 기준선
    inn = df.sort_values("net_flow_per_day", ascending=False).head(top_n)
    out = df.sort_values("net_flow_per_day").head(top_n)
    d = pd.concat([inn, out]).drop_duplicates("station_id").sort_values("net_flow_per_day", ascending=False).reset_index(drop=True)
    names = d["station_name"].fillna(d["station_id"]).astype(str).str.slice(0, 16)
    colors = np.where(d["net_flow_per_day"] >= 0, BLUE, RED)
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(d) + 1.8))
    ax.barh(range(len(d)), d["net_flow_per_day"], color=colors, height=0.68, edgecolor=SURFACE, linewidth=2)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(names, fontsize=9); ax.invert_yaxis()
    ax.grid(axis="y", visible=False); ax.margins(x=0.1)
    for i in sorted({0, len(d) - 1}):                     # 양 끝 값만 직접 라벨
        v = d.loc[i, "net_flow_per_day"]
        ax.annotate(f"{v:+,.2f}" if abs(v) < 1 else f"{v:+,.1f}", (v, i), xytext=(5 if v >= 0 else -5, 0), textcoords="offset points",
                    ha="left" if v >= 0 else "right", va="center", color=INK2, fontsize=9)
    ax.set_xlabel("평일 하루당 순유입 (반납 - 대여, 건)")
    ax.legend(handles=[Patch(color=BLUE, label="순유입 (반납 > 대여)"), Patch(color=RED, label="순유출 (대여 > 반납)")],
              frameon=False, loc="lower right", labelcolor=INK2)
    _title(ax, f"출근시간대({hours_label}) 평일 순유입·순유출 상위 {top_n}곳씩", "순유입 = 자전거가 쌓이는 곳, 순유출 = 빠져나가는 곳")
    fig.tight_layout(); plt.show()


def plot_bar_h(series, title, sub, xlabel, color=BLUE, top=None):
    # 가로 막대 (값 큰 순, 끝 값만 라벨)
    s = series.sort_values(ascending=False)
    if top:
        s = s.head(top)
    fig, ax = plt.subplots(figsize=(8, 0.3 * len(s) + 1.6))
    ax.barh(range(len(s)), s.values, color=color, height=0.68, edgecolor=SURFACE, linewidth=2)
    ax.set_yticks(range(len(s))); ax.set_yticklabels([str(i)[:16] for i in s.index], fontsize=9); ax.invert_yaxis()
    ax.grid(axis="y", visible=False); ax.margins(x=0.08)
    ax.xaxis.set_major_formatter(_thousands); ax.set_xlabel(xlabel)
    ax.annotate(f"{s.iloc[0]:,.0f}", (s.iloc[0], 0), xytext=(5, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
    _title(ax, title, sub)
    fig.tight_layout(); plt.show()

print("그래프 함수 준비 완료")

# 1. 기본 EDA
## 1-1. 시간대별 전체 대여량/반납량
**왜:** 하루 중 언제 이용이 몰리는지(출퇴근 피크) 확인합니다.

| 컬럼 | 뜻 |
|---|---|
| `rent_total`, `return_total` | 기간 전체 시간대별 대여·반납 합계 |
| `avg_rent_per_day`, `avg_return_per_day` | 하루 평균 (= 합계 ÷ 기간 일수) |

In [ ]:
df_hourly_all = (df_A.groupby("hour", as_index=False)[["rent_count", "return_count"]].sum()
                 .rename(columns={"rent_count": "rent_total", "return_count": "return_total"}))
df_hourly_all["avg_rent_per_day"]   = (df_hourly_all["rent_total"]   / DAYS_IN_PERIOD).round(1)
df_hourly_all["avg_return_per_day"] = (df_hourly_all["return_total"] / DAYS_IN_PERIOD).round(1)
display(df_hourly_all)
plot_hourly_all(df_hourly_all)

## 1-2. 평일·주말·공휴일 시간대별 패턴
**왜:** 평일에만 출퇴근 피크가 있으면 통근형 수요 → 출근시간(`COMMUTE_HOURS`) 기준을 정합니다.

| 컬럼 | 뜻 |
|---|---|
| `day_type` | weekday 평일 / weekend 주말 / holiday 평일 공휴일 |
| `n_days` | 그 요일구분의 일수 (평균의 분모) |
| `avg_rent_per_day`, `avg_return_per_day` | 요일구분별 하루 평균 |

In [ ]:
df_hourly_daytype = (df_A.groupby(["day_type", "hour"], as_index=False)[["rent_count", "return_count"]].sum()
                     .rename(columns={"rent_count": "rent_total", "return_count": "return_total"}))
df_hourly_daytype["n_days"] = df_hourly_daytype["day_type"].map(DAYS_BY_TYPE).astype("int64")
df_hourly_daytype["avg_rent_per_day"]   = (df_hourly_daytype["rent_total"]   / df_hourly_daytype["n_days"]).round(1)
df_hourly_daytype["avg_return_per_day"] = (df_hourly_daytype["return_total"] / df_hourly_daytype["n_days"]).round(1)
plot_hourly_daytype(df_hourly_daytype)

# 평일 대여·반납 피크 상위 5시간 → COMMUTE_HOURS 결정에 사용
wk = df_hourly_daytype[df_hourly_daytype["day_type"] == "weekday"]
display(wk.nlargest(5, "avg_rent_per_day")[["hour", "avg_rent_per_day"]])
display(wk.nlargest(5, "avg_return_per_day")[["hour", "avg_return_per_day"]])

## 1-3·1-4. 대여소별 대여량/반납량, net_flow
**왜:** 대여가 많은(비기 쉬운) 대여소와 반납이 많은(넘치기 쉬운) 대여소를 찾습니다.

| 컬럼 | 뜻 |
|---|---|
| `net_flow` | 반납 − 대여 (음수 = 순유출, 양수 = 순유입) |
| `rent_per_month`, `return_per_month` | 월평균 (= 합계 ÷ 개월 수) |
| `MIN_TRIPS` | 이용이 너무 적은 대여소는 산점도에서 제외 (기본 1,000건) |

In [ ]:
MIN_TRIPS = 1000          # 이용이 너무 적은 대여소는 그래프에서 제외
df_station_flow = df_A.groupby("station_id", as_index=False).agg(rent_total=("rent_count", "sum"), return_total=("return_count", "sum"))
df_station_flow["net_flow"]         = df_station_flow["return_total"] - df_station_flow["rent_total"]
df_station_flow["rent_per_month"]   = (df_station_flow["rent_total"]   / N_MONTHS).round(1)
df_station_flow["return_per_month"] = (df_station_flow["return_total"] / N_MONTHS).round(1)
print("전체 대여소:", len(df_station_flow))
busy = df_station_flow[(df_station_flow["rent_total"] + df_station_flow["return_total"]) >= MIN_TRIPS]
if len(busy):
    plot_station_scatter(busy)
else:
    print(f"이용 {MIN_TRIPS:,}건 이상인 대여소가 없어 그래프를 생략합니다.")
display(df_station_flow.sort_values("net_flow").head(10))     # 순유출 상위
display(df_station_flow.sort_values("net_flow").tail(10))     # 순유입 상위

## 1-5·1-6. 출근시간대 순유출/순유입 상위 대여소
**왜:** 평일 07~09시에 자전거가 빠져나가는(빨강) / 쌓이는(파랑) 대여소 = 재배치 우선 후보입니다.

| 컬럼 | 뜻 |
|---|---|
| `net_flow_per_day` | 평일 하루당 (반납 − 대여), 출근시간 합 |
| `station_name` | 대여소 이름 (`bike_station_both`와 결합, 못 찾으면 ID 표시) |

In [ ]:
COMMUTE_HOURS = [7, 8, 9]          # 1-2 평일 피크를 보고 정한 출근시간
TOP_N = 15
n_weekdays = int(DAYS_BY_TYPE.get("weekday", 0))

df_commute = (df_A[(df_A["day_type"] == "weekday") & df_A["hour"].isin(COMMUTE_HOURS)]
              .groupby("station_id", as_index=False)[["rent_count", "return_count"]].sum()
              .rename(columns={"rent_count": "rent_total", "return_count": "return_total"}))
df_commute = df_commute[(df_commute["rent_total"] + df_commute["return_total"]) >= MIN_TRIPS]
df_commute["net_flow_per_day"] = ((df_commute["return_total"] - df_commute["rent_total"]) / max(n_weekdays, 1)).round(2)
df_commute = df_commute.merge(df_both[["rent_id", "rent_nm"]].rename(columns={"rent_id": "station_id", "rent_nm": "station_name"}),
                              on="station_id", how="left", validate="many_to_one")
print("이름을 못 찾은 대여소 수:", int(df_commute["station_name"].isna().sum()), "(0이면 결합 완벽)")
if len(df_commute):
    plot_commute_netflow(df_commute, TOP_N, f"{COMMUTE_HOURS[0]:02d}~{COMMUTE_HOURS[-1]:02d}시")
else:
    print(f"출근시간 이용 {MIN_TRIPS:,}건 이상인 대여소가 없어 그래프를 생략합니다.")

## 1-7. 집계 결과 검증
**왜:** 그래프 숫자가 집계표 합계와 맞는지 확인합니다.
- 시간대별 하루 평균의 합 = 전체 합계 ÷ 기간 일수
- `net_flow` 합계는 0이 아닐 수 있습니다: 반납 누락·이상 기록은 `rent_count`에는 들어가고 `return_count`에는 안 들어가기 때문 (01 노트북 4-3)

In [ ]:
rent_total, ret_total = int(df_A["rent_count"].sum()), int(df_A["return_count"].sum())
print("하루 평균 총 대여:", round(df_hourly_all["avg_rent_per_day"].sum()), "/ 직접 계산:", round(rent_total / DAYS_IN_PERIOD))
print("하루 평균 총 반납:", round(df_hourly_all["avg_return_per_day"].sum()), "/ 직접 계산:", round(ret_total / DAYS_IN_PERIOD))
actual_flow = int(df_station_flow["net_flow"].sum())
print(f"net_flow 합계 = {actual_flow:,} / 기대값(반납 − 대여) = {ret_total - rent_total:,}")
assert actual_flow == ret_total - rent_total, "순유입 집계가 반납−대여와 다릅니다."

# 2. 지하철 연관 EDA
- 대여소별 **가장 가까운 역**(`station_subway_link.is_nearest`)의 승하차와 그 대여소의 대여·반납을 같은 연월·시간끼리 연결
- 승하차는 월 합계라 그 달 일수로 나눠 **하루 평균**으로 맞춤 (따릉이도 하루 평균)

| 비교 | 뜻 |
|---|---|
| 역 승차 ↔ 대여 | 역에서 타는 사람이 많을 때 근처 대여도 많은가 |
| 역 하차 ↔ 반납 | 역에서 내리는 사람이 많을 때 근처 반납도 많은가 |
| 역까지 거리 ↔ 대여·반납 | 역에 가까울수록 이용이 많은가 |

## 2-1. 시간대별 상관: 역 승차 ↔ 대여, 역 하차 ↔ 반납
**왜:** 시간을 섞으면 의미가 흐려지므로 0~23시마다 따로 Pearson 상관을 봅니다. 점 = (대여소, 연월, 시간).

| 컬럼 | 뜻 |
|---|---|
| `n` | 점 개수 |
| `corr_rent_on` | 역 승차 ↔ 대여소 대여 상관계수 (+1에 가까울수록 같이 오르내림) |
| `corr_ret_off` | 역 하차 ↔ 대여소 반납 상관계수 |

In [ ]:
bike = df_A.groupby(["station_id", "ym", "hour"], as_index=False)[["rent_count", "return_count"]].sum()
jj = (df_link[df_link["is_nearest"]][["rent_id", "phys_key"]].rename(columns={"rent_id": "station_id"})
      .merge(bike, on="station_id", how="inner", validate="one_to_many")
      .merge(df_subway_ride, on=["phys_key", "ym", "hour"], how="inner", validate="many_to_one")
      .merge(days_ym, left_on="ym", right_index=True, how="left", validate="many_to_one"))
for src, dst in [("rent_count", "rent_pd"), ("return_count", "ret_pd"), ("get_on", "on_pd"), ("get_off", "off_pd")]:
    jj[dst] = jj[src].astype(float) / jj["days_in_month"]          # 하루 평균으로 맞춤

rows = []
for h, g in jj.groupby("hour"):
    rows.append({"hour": h, "n": len(g), "corr_rent_on": g["rent_pd"].corr(g["on_pd"]), "corr_ret_off": g["ret_pd"].corr(g["off_pd"])})
df_corr_hour = pd.DataFrame(rows, columns=["hour", "n", "corr_rent_on", "corr_ret_off"])
display(df_corr_hour.round(3))

- 위 표를 선그래프로: 파랑 = 승차↔대여, 주황 = 하차↔반납, 0보다 위면 양의 관계

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.axhline(0, color=INK2, lw=1)                                     # 0 기준선
ax.plot(df_corr_hour["hour"], df_corr_hour["corr_rent_on"], color=BLUE,   lw=2, label="역 승차 ↔ 대여소 대여")
ax.plot(df_corr_hour["hour"], df_corr_hour["corr_ret_off"], color=ORANGE, lw=2, label="역 하차 ↔ 대여소 반납")
ax.set_xticks(range(0, 24, 2)); ax.set_xlim(-0.5, 23.5)
ax.set_xlabel("시간(시)"); ax.set_ylabel("상관계수")
ax.legend(frameon=False, loc="best", labelcolor=INK2)
_title(ax, "시간대별 지하철 승하차와 인근 대여소 이용의 상관", "점 = (대여소, 연월, 시간) · 가장 가까운 역 기준 · 하루 평균")
fig.tight_layout(); plt.show()

## 2-2. 대여소 단위 비교 (Spearman, 4가지 조합)
**왜:** 월 변동을 빼고 "이용이 많은 역 근처 대여소가 이용도 많은가"를 순위 상관으로 봅니다 (큰 대여소 몇 곳의 영향을 줄임).

| 컬럼 | 뜻 |
|---|---|
| `rent_on`, `rent_off` | 대여 ↔ 역 승차 / 역 하차 |
| `ret_on`, `ret_off` | 반납 ↔ 역 승차 / 역 하차 |
| `MAX_M` | 가장 가까운 역까지 최대 거리 (300으로 바꾸면 역세권만) |

- 해석: 아침 반납↔승차가 높으면 "따릉이로 역에 와서 반납 후 승차", 저녁 대여↔하차가 높으면 "역에서 내려 대여" 흐름

In [ ]:
MAX_M = 500
st_ids = df_A["station_id"][smatch(df_A["station_id"], r"^ST-")].unique()
bike_h = (df_A[df_A["station_id"].isin(st_ids)].groupby(["station_id", "hour"])[["rent_count", "return_count"]].sum()
          .div(DAYS_IN_PERIOD).rename(columns={"rent_count": "rent_pd", "return_count": "ret_pd"}).reset_index())
days_sub = int(days_ym[days_ym.index.isin(df_subway_ride["ym"].unique())].sum())       # 지하철 자료가 있는 달의 일수
sub_h = (df_subway_ride.groupby(["phys_key", "hour"])[["get_on", "get_off"]].sum().div(days_sub)
         .rename(columns={"get_on": "on_pd", "get_off": "off_pd"}).reset_index())
lk = df_link[df_link["is_nearest"] & (df_link["distance_m"] <= MAX_M)][["rent_id", "phys_key"]].rename(columns={"rent_id": "station_id"})
j2 = (lk.merge(bike_h, on="station_id", how="inner", validate="one_to_many")
        .merge(sub_h, on=["phys_key", "hour"], how="inner", validate="many_to_one"))
rows = []
for h, g in j2.groupby("hour"):
    r = g[["rent_pd", "ret_pd", "on_pd", "off_pd"]].corr(method="spearman")
    rows.append({"hour": h, "n": len(g), "rent_on": r.loc["rent_pd", "on_pd"], "rent_off": r.loc["rent_pd", "off_pd"],
                 "ret_on": r.loc["ret_pd", "on_pd"], "ret_off": r.loc["ret_pd", "off_pd"]})
df_corr2 = pd.DataFrame(rows, columns=["hour", "n", "rent_on", "rent_off", "ret_on", "ret_off"])
display(df_corr2.round(3))

# 왼쪽 = 대여와의 상관, 오른쪽 = 반납과의 상관 / 파랑 = 역 승차, 주황 = 역 하차
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, (c_on, c_off, ttl) in zip(axes, [("rent_on", "rent_off", "대여소 대여와의 상관"), ("ret_on", "ret_off", "대여소 반납과의 상관")]):
    ax.axhline(0, color=INK2, lw=1)
    ax.plot(df_corr2["hour"], df_corr2[c_on],  color=BLUE,   lw=2, label="역 승차")
    ax.plot(df_corr2["hour"], df_corr2[c_off], color=ORANGE, lw=2, label="역 하차")
    ax.set_xticks(range(0, 24, 3)); ax.set_xlim(-0.5, 23.5)
    ax.set_xlabel("시간(시)"); ax.set_title(ttl, loc="left", color=INK, fontsize=11, fontweight="bold")
axes[0].set_ylabel("Spearman 상관계수"); axes[0].legend(frameon=False, loc="best", labelcolor=INK2)
fig.suptitle(f"대여소 단위 비교 (가장 가까운 역 {MAX_M}m 이내, 하루 평균)", x=0.01, ha="left", color=INK, fontsize=13, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

## 2-3. 역과의 거리 구간별 평균 대여·반납 (역세권 vs 비역세권)
**왜:** 역에 가까울수록 이용이 많은지 구간으로 비교합니다. 전체 시간과 출근시간(07~09시)을 따로 봅니다.

| 컬럼 | 뜻 |
|---|---|
| `band` | 가장 가까운 역까지 0-300m / 300-500m / 500m 초과(연결 없음) |
| `n_stations` | 대여소 수 |
| `avg_rent_all` | 하루 평균 대여 (전체 시간) |
| `avg_rent_commute`, `avg_ret_commute` | 출근시간 하루 평균 대여·반납 |
| `median_rent_all` | 하루 평균 대여의 중앙값 |

- 주의: 기존 3_eda와 같은 정의라 좌표 없는 대여소도 '500m 초과'에 섞여 있음 → 2-6에서 빼고 다시 계산

In [ ]:
commute = df_A["hour"].between(7, 9)
st = (df_A[df_A["station_id"].isin(st_ids)]
      .assign(rent_c=lambda d: d["rent_count"].where(commute, 0), ret_c=lambda d: d["return_count"].where(commute, 0))
      .groupby("station_id").agg(rent_all=("rent_count", "sum"), rent_commute=("rent_c", "sum"), ret_commute=("ret_c", "sum"))
      .div(DAYS_IN_PERIOD).reset_index())
st = st.merge(df_link[df_link["is_nearest"]][["rent_id", "radius_band", "distance_m"]].rename(columns={"rent_id": "station_id"}),
              on="station_id", how="left", validate="one_to_one")
st["band"] = st["radius_band"].fillna("500m 초과(연결 없음)")


def band_table(d):
    # 거리 구간별 대여소 수 · 평균 · 중앙값
    return (d.groupby("band").agg(n_stations=("station_id", "size"), avg_rent_all=("rent_all", "mean"),
                                  avg_rent_commute=("rent_commute", "mean"), avg_ret_commute=("ret_commute", "mean"),
                                  median_rent_all=("rent_all", "median")).round(1).reset_index())


df_band = band_table(st)
display(df_band)

## 2-4. 같은 구 안에서 역세권 vs 비역세권
**왜:** 번화가 vs 외곽 같은 지역 차이를 줄이려고 **구 안에서** 비교합니다. 역세권 = 가장 가까운 역 300m 이내, 비역세권 = 그 외(연결 없음 포함).

| 컬럼 | 뜻 |
|---|---|
| `n_near`, `n_far` | 역세권 / 비역세권 대여소 수 |
| `rent_ratio` | 역세권 ÷ 비역세권 하루 평균 대여 (1보다 크면 역세권이 많음) |
| `ret_commute_ratio` | 역세권 ÷ 비역세권 출근시간 반납 |

In [ ]:
st_gu = st.merge(df_both[["rent_id", "sta_loc"]].rename(columns={"rent_id": "station_id"}), on="station_id", how="inner", validate="one_to_one")
st_gu["grp"] = np.where(st_gu["distance_m"] <= NEAR_RADIUS_M, "near", "far")
p = st_gu.groupby(["sta_loc", "grp"]).agg(n=("station_id", "size"), avg_rent_all=("rent_all", "mean"),
                                          avg_ret_commute=("ret_commute", "mean")).unstack("grp")
p.columns = [f"{a}_{b}" for a, b in p.columns]
p.index.name = "gu"
ratio_ok = {"avg_rent_all_near", "avg_rent_all_far"} <= set(p.columns)
if ratio_ok:
    p["rent_ratio"] = (p["avg_rent_all_near"] / p["avg_rent_all_far"]).round(2)
    p["ret_commute_ratio"] = (p["avg_ret_commute_near"] / p["avg_ret_commute_far"]).round(2)
    p = p.dropna(subset=["rent_ratio"]).sort_values("rent_ratio", ascending=False)
    display(p[["n_near", "n_far", "rent_ratio", "ret_commute_ratio"]])
    print(f"구 {len(p)}곳 중 역세권 대여가 더 많은 구: {(p['rent_ratio'] > 1).sum()}곳 (중앙값 {p['rent_ratio'].median():.2f}배)")
    print(f"구 {len(p)}곳 중 역세권 출근 반납이 더 많은 구: {(p['ret_commute_ratio'] > 1).sum()}곳 (중앙값 {p['ret_commute_ratio'].median():.2f}배)")
else:
    print("역세권/비역세권 중 한쪽이 없는 구만 있어 비교할 수 없습니다.")

## 2-5. 좌표 없는 대여소의 영향
**왜:** 좌표가 0·결측인 대여소는 역 거리를 계산할 수 없어(`has_coord=False`) 역세권 변수가 결측입니다. 그 비중이 작으면(1~2% 이하) 빼고 분석해도 결론에 큰 영향이 없습니다.

| 컬럼 | 뜻 |
|---|---|
| `n_stations_no_coord` | 좌표 없는 대여소 수 |
| `rent_no_coord`, `pct_of_all_rent` | 그 대여소들의 대여 합계, 전체 대여 중 비중(%) |

In [ ]:
st_rent = (df_A[smatch(df_A["station_id"], r"^ST-[0-9]+$")].groupby("station_id")["rent_count"].sum()
           .rename("rent_total").reset_index())
no_coord = feature_station_subway[~feature_station_subway["has_coord"]].merge(st_rent, on="station_id", how="left", validate="one_to_one")
display(pd.DataFrame({"n_stations_no_coord": [len(no_coord)],
                      "rent_no_coord": [int(no_coord["rent_total"].fillna(0).sum())],
                      "pct_of_all_rent": [round(100 * no_coord["rent_total"].fillna(0).sum() / max(st_rent["rent_total"].sum(), 1), 2)]}))

# 이용량 상위 15곳
display(no_coord.dropna(subset=["rent_total"]).sort_values("rent_total", ascending=False)
        [["station_id", "station_name", "gu", "rent_total"]].head(15).reset_index(drop=True))

## 2-6. 거리 구간별 평균 이용 재계산 (좌표 없는 대여소 제외)
**왜:** 2-3에서는 좌표 없는 대여소가 '500m 초과'에 섞여 있었으므로 `has_coord=True`인 대여소만으로 다시 계산해 결론이 바뀌는지 봅니다.

- 구간: `nearest_distance_m` 기준 0-300m / 300-500m / 500m 초과 (500m 안에 역이 없으면 500m 초과)

In [ ]:
st2 = st.drop(columns=["band", "radius_band", "distance_m"]).merge(
    feature_station_subway[["station_id", "has_coord", "nearest_distance_m"]], on="station_id", how="inner", validate="one_to_one")
st2 = st2[st2["has_coord"]].copy()
st2["band"] = np.select([st2["nearest_distance_m"] <= NEAR_RADIUS_M, st2["nearest_distance_m"] <= MAX_LINK_RADIUS_M],
                        [f"0-{NEAR_RADIUS_M}m", f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m"], default="500m 초과")
df_band2 = band_table(st2)
display(df_band2)
print(f"2-3 대비 제외된 대여소: {len(st) - len(st2):,}곳 (좌표 없음 또는 대여소 표에 없음)")

## 2-7. (추가) 대여소 변수 상관 행렬
**왜:** 대여소 1곳 = 1점으로, 이용량과 지하철 변수(거리·역 수·역 승하차)가 서로 얼마나 같이 움직이는지 한 장으로 봅니다 (Spearman, 좌표 정상 대여소만).

| 변수 | 뜻 |
|---|---|
| `rent_all`, `rent_commute`, `ret_commute` | 하루 평균 대여, 출근시간 대여·반납 |
| `nearest_distance_m` | 가장 가까운 역 거리 (거리 제한 없음, `bike_station_nearest`) |
| `n_subway_300`, `n_subway_500` | 300m / 500m 안 역 수 |
| `hold_num` | 거치대 수 |
| `subway_on_pd`, `subway_off_pd` | 가장 가까운 역 하루 평균 승차·하차 (500m 안 역이 없으면 결측 → 그 쌍만 빠짐) |

In [ ]:
sub_tot = (df_subway_ride.groupby("phys_key")[["get_on", "get_off"]].sum().div(days_sub)
           .rename(columns={"get_on": "subway_on_pd", "get_off": "subway_off_pd"}).reset_index())
mat = (st[["station_id", "rent_all", "rent_commute", "ret_commute"]]
       .merge(feature_station_subway[["station_id", "has_coord", "hold_num", "n_subway_300", "n_subway_500", "nearest_phys_key"]],
              on="station_id", how="inner", validate="one_to_one")
       .merge(df_nearest[["rent_id", "nearest_any_m"]].rename(columns={"rent_id": "station_id", "nearest_any_m": "nearest_distance_m"}),
              on="station_id", how="left", validate="one_to_one")
       .merge(sub_tot.rename(columns={"phys_key": "nearest_phys_key"}), on="nearest_phys_key", how="left", validate="many_to_one"))
mat = mat[mat["has_coord"]]
cols = ["rent_all", "rent_commute", "ret_commute", "nearest_distance_m", "n_subway_300", "n_subway_500", "hold_num",
        "subway_on_pd", "subway_off_pd"]
corr_m = mat[cols].astype(float).corr(method="spearman")
display(corr_m.round(2))

fig, ax = plt.subplots(figsize=(8.6, 7.2))
im = ax.imshow(corr_m.values, cmap="RdBu_r", vmin=-1, vmax=1)        # 빨강 = 양의 상관, 파랑 = 음의 상관
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=9)
ax.grid(False)
for i in range(len(cols)):
    for j in range(len(cols)):
        v = corr_m.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if abs(v) > 0.6 else INK)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Spearman 상관계수")
_title(ax, "대여소 변수 상관 행렬", f"대여소 {len(mat):,}곳 (좌표 정상) · {PERIOD_LABEL}")
fig.tight_layout(); plt.show()

## 2-8. 최종 EDA 결론
### (1) 이번 기간 결과 자동 요약
**왜:** 위 결과 수치를 가설별로 모아 팀 검토에 씁니다. 기간이 바뀌면 수치도 바뀝니다 (화면 표시만, 파일 저장 없음).

In [ ]:
def _get(df, band, col):
    v = df.loc[df["band"] == band, col]
    return float(v.iloc[0]) if len(v) else float("nan")

def _rng(s):
    return f"{s.min():.2f}~{s.max():.2f}" if s.notna().any() else "계산 불가"

near_b, mid_b, far_b = f"0-{NEAR_RADIUS_M}m", f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m", "500m 초과(연결 없음)"
morning = df_corr2[df_corr2["hour"].between(5, 7)]
evening = df_corr2[df_corr2["hour"].between(19, 23)]
peak    = df_corr2[df_corr2["hour"].between(8, 9)]
other   = df_corr2[~df_corr2["hour"].between(8, 9) & df_corr2["hour"].between(6, 23)]
summary = pd.DataFrame([
    ["역과 가까울수록 대여가 많다",
     f"{near_b} {_get(df_band, near_b, 'avg_rent_all'):.1f} / {mid_b} {_get(df_band, mid_b, 'avg_rent_all'):.1f} / 500m 초과 {_get(df_band, far_b, 'avg_rent_all'):.1f}"
     + (f" · 구별 비교 {int((p['rent_ratio'] > 1).sum())}/{len(p)}개 구에서 역세권이 높음" if ratio_ok else ""),
     "distance_to_subway, is_subway_near_300"],
    ["출근시간 반납은 역 근처에서 특히 많다",
     f"출근 07~09시 반납 {near_b} {_get(df_band, near_b, 'avg_ret_commute'):.1f} vs 500m 초과 {_get(df_band, far_b, 'avg_ret_commute'):.1f}"
     + (f" · {int((p['ret_commute_ratio'] > 1).sum())}/{len(p)}개 구" if ratio_ok else ""),
     "distance_to_subway(출근 반납)"],
    ["아침 역 승차가 많으면 인근 반납·대여가 늘어난다",
     f"5~7시 반납↔승차 {_rng(morning['ret_on'])}, 대여↔승차 {_rng(morning['rent_on'])}", "subway_boarding(아침)"],
    ["저녁 역 하차가 많으면 인근 대여가 늘어난다", f"19~23시 대여↔하차 {_rng(evening['rent_off'])}", "subway_alighting(저녁)"],
    ["출근시간 하차가 많으면 인근 반납이 늘어난다", f"8~9시 반납↔하차 {_rng(peak['ret_off'])}, 그 외 {_rng(other['ret_off'])}", "subway_alighting"],
    ["출근시간에 지하철 효과가 강해진다",
     f"출근 대여 {near_b} {_get(df_band, near_b, 'avg_rent_commute'):.1f} vs 500m 초과 {_get(df_band, far_b, 'avg_rent_commute'):.1f}",
     "is_commute_morning, is_commute_evening"],
], columns=["가설", "EDA 결과", "변수 후보"])
summary.insert(1, "분석 기간", PERIOD_LABEL)
summary["판단"] = "결과를 보고 팀에서 기입"
display(summary)

### (2) 기존 결론 (참고: 2021-01 ~ 2025-12, 기존 3_eda "6. 최종 EDA 결론")
> 이전 기간·이전 집계 정의(정상 완료 거래 기준) 결과입니다. 위 자동 요약과 **비교용**으로만 보세요.

|가설|EDA 결과 (2021~2025)|변수 후보|판단|
|--|--|--|--|
|역과 가까울수록 대여가 많다|0~300m 44.1 > 300~500m 35.0 > 500m 초과 29.0 (중앙값도 같은 순서). 같은 구 안에서 비교해도 25개 구 모두 역세권이 높음(중앙값 1.49배)|distance_to_subway, is_subway_near(300m)|채택|
|출근시간 반납은 역 근처에서 특히 많다|출근 07~09시 반납 역세권 7.2 vs 4.3(+67%). 구별 비교에서도 25개 중 23개 구에서 역세권이 높음(중앙값 1.70배)|distance_to_subway(출근 반납)|채택|
|아침 역 승차가 많으면 인근 반납·대여가 늘어난다|5~7시 반납↔승차 0.19~0.28, 대여↔승차 0.17~0.24 (대여소 단위 Spearman)|subway_boarding(아침)|채택 후보(약~보통)|
|저녁 역 하차가 많으면 인근 대여가 늘어난다|19~23시 대여↔하차 0.21~0.27|subway_alighting(저녁)|채택 후보(약~보통)|
|출근시간 하차가 많으면 인근 반납이 늘어난다|8~9시 반납↔하차 0.22~0.25, 그 외 시간은 0.17~0.20으로 평탄|subway_alighting|보류|
|출근시간에 지하철 효과가 강해진다|대여 격차는 작음(6.4 vs 5.5), 반납 격차는 큼|is_commute_time(07~09시), is_evening_commute(17~19시)|부분 채택|

- 해석 주의: 상관·비교는 인과가 아님. 역 근처는 원래 번화가라 입지 효과가 섞여 있음
- 지하철 변수 단독 설명력은 낮음(상관 최대 약 0.28) → 날씨·유동인구·건물 등과 함께 쓰는 보조 변수
- 중구·성동구는 출근 반납에서 역세권이 낮게 나옴(도심 전체가 번화가라 구분이 흐림)

# 3. 추가 시각화 (3_eda에 없던 그래프)
## 3-1. 대여소 분포
**왜:** 구별 대여소 수·거치대 규모·위치를 보고 데이터가 서울 전역을 고르게 덮는지 확인합니다.

| 컬럼 | 뜻 |
|---|---|
| `sta_loc` | 자치구 |
| `hold_num` | 거치대 수 |
| `sta_lat`, `sta_long` | 위도, 경도 (`feature_station_subway`, 좌표 정상만) |

In [ ]:
plot_bar_h(df_both["sta_loc"].value_counts(), "구별 대여소 수", f"bike_station_both {len(df_both):,}곳 · 수집 시점 기준", "대여소 수")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
hn = df_both["hold_num"].dropna()
vc = hn.value_counts().sort_index()
axes[0].bar(vc.index.astype(int), vc.values, color=BLUE, width=0.8, edgecolor=SURFACE)
axes[0].set_xlabel("거치대 수"); axes[0].set_ylabel("대여소 수")
axes[0].yaxis.set_major_locator(MaxNLocator(integer=True)); axes[0].yaxis.set_major_formatter(_thousands)
_title(axes[0], "대여소별 거치대 수 분포", f"결측 {df_both['hold_num'].isna().sum()}곳 · 중앙값 {hn.median():.0f}")
loc = feature_station_subway[feature_station_subway["has_coord"]]          # 위경도 교정 후 좌표
axes[1].scatter(loc["sta_long"], loc["sta_lat"], s=4, color=BLUE, alpha=0.5, edgecolor="none", label="대여소")
pad = 0.01                                                                   # 대여소 범위 밖 역(수도권)은 그리지 않음
sub = df_subway_station[df_subway_station["lat"].between(loc["sta_lat"].min() - pad, loc["sta_lat"].max() + pad)
                        & df_subway_station["lot"].between(loc["sta_long"].min() - pad, loc["sta_long"].max() + pad)]
axes[1].scatter(sub["lot"], sub["lat"], s=10, color=ORANGE, marker="s", label="지하철역")
axes[1].set_xlabel("경도"); axes[1].set_ylabel("위도"); axes[1].set_aspect(1 / np.cos(np.radians(37.55)))
axes[1].legend(frameon=False, loc="lower left", labelcolor=INK2)
_title(axes[1], "대여소·지하철역 위치", f"대여소 {len(loc):,}곳 (좌표 없음 {(~feature_station_subway['has_coord']).sum()}곳 제외) · 범위 안 물리역 {len(sub):,}곳")
fig.tight_layout(); plt.show()

## 3-2. 가장 가까운 지하철역까지 거리 분포
**왜:** 300m·500m 기준이 대여소를 적당히 나누는지 확인합니다 (`nearest_any_m` = 거리 제한 없이 가장 가까운 역).

In [ ]:
nearest_m = df_nearest["nearest_any_m"]
for r in (NEAR_RADIUS_M, MAX_LINK_RADIUS_M):
    print(f"가장 가까운 역이 {r}m 이내인 대여소: {(nearest_m <= r).mean():.1%}")
display(nearest_m.describe().round(0).to_frame("nearest_any_m"))
print("역당 근처(500m) 대여소 수 분포")
display(df_link.groupby("phys_key").size().describe().round(1).to_frame("대여소 수"))

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.hist(nearest_m.clip(upper=2000), bins=np.arange(0, 2050, 50), color=BLUE, edgecolor=SURFACE, linewidth=1)
for r, yf in ((NEAR_RADIUS_M, 0.92), (MAX_LINK_RADIUS_M, 0.80)):
    ax.axvline(r, color=INK2, lw=1, ls="--")
    ax.annotate(f"{r}m 이내 {(nearest_m <= r).mean():.0%}", (r, ax.get_ylim()[1] * yf), xytext=(4, 0),
                textcoords="offset points", color=INK2, fontsize=9)
ax.set_xlabel("가장 가까운 지하철역까지 거리 (m, 2km 이상은 2km에 합침)"); ax.set_ylabel("대여소 수")
_title(ax, "대여소에서 가장 가까운 지하철역까지 거리", f"대여소 {len(nearest_m):,}곳 · 물리역 {len(df_subway_station):,}곳")
fig.tight_layout(); plt.show()

## 3-3. 지하철 승하차 추이
**왜:** 월별 규모 변화(급감한 달 = 수집 누락 의심)와 시간대별 하루 평균 승하차 모양을 확인합니다.

| 컬럼 | 뜻 |
|---|---|
| `get_on`, `get_off` | 매칭된 물리역 합계 승차·하차 |

In [ ]:
mon = df_subway_ride.groupby("ym")[["get_on", "get_off"]].sum()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
x = pd.to_datetime(mon.index, format="%Y%m")
axes[0].plot(x, mon["get_on"], color=BLUE, lw=2, marker="o", ms=3, label="승차")
axes[0].plot(x, mon["get_off"], color=ORANGE, lw=2, marker="o", ms=3, label="하차")
axes[0].set_ylim(bottom=0); axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e6:,.1f}백만"))
axes[0].legend(frameon=False, labelcolor=INK2); axes[0].set_ylabel("월 합계 인원")
axes[0].xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=8))
axes[0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axes[0].xaxis.get_major_locator()))
_title(axes[0], "월별 지하철 승하차 (매칭된 물리역 합계)", f"{df_subway_ride['phys_key'].nunique()}개 물리역")
hr = df_subway_ride.groupby("hour")[["get_on", "get_off"]].sum() / days_sub
axes[1].plot(hr.index, hr["get_on"], color=BLUE, lw=2, label="승차")
axes[1].plot(hr.index, hr["get_off"], color=ORANGE, lw=2, label="하차")
axes[1].set_xticks(range(0, 24, 3)); axes[1].set_xlabel("시간(시)"); axes[1].set_ylabel("하루 평균 인원")
axes[1].yaxis.set_major_formatter(_thousands); axes[1].set_ylim(bottom=0)
_title(axes[1], "시간대별 하루 평균 승하차", "모든 물리역 합계")
fig.tight_layout(); plt.show()
print("전월 대비 30% 넘게 줄어든 달:", list(mon.index[mon["get_on"].pct_change() < -0.3]) or "없음")

## 3-4. 일별 추정 승하차 확인 (`subway_ride_day_hour_full`, 있을 때만)
**왜:** 월 합계를 날짜별로 나눈 결과가 평일·주말 모양을 제대로 반영하는지, 실측/추정 비율이 얼마인지 확인합니다.

| 컬럼 | 뜻 |
|---|---|
| `shape_source` | `measured` 실측 / `allocated_range` 덩어리 시간 배분 / `profile` 요일 프로필 추정 |
| `day_type` | 요일구분 (달력과 결합) |

In [ ]:
if len(df_subway_full):
    display(df_subway_full["shape_source"].value_counts().to_frame("행 수"))
    f = df_subway_full.merge(cal[["cal_date", "day_type"]], left_on="ride_date", right_on="cal_date", how="left", validate="many_to_one")
    prof = f.groupby(["day_type", "hour"])["get_on"].sum() / f.groupby("day_type")["ride_date"].nunique()
    fig, ax = plt.subplots(figsize=(9, 4.4))
    for key, name, color in [("weekday", "평일", BLUE), ("weekend", "주말", ORANGE), ("holiday", "공휴일", AQUA)]:
        if key in prof.index.get_level_values(0):
            ax.plot(prof.loc[key].index, prof.loc[key].values, color=color, lw=2, label=name)
    ax.set_xticks(range(0, 24, 3)); ax.set_xlim(-0.5, 23.5); ax.set_ylim(bottom=0)
    ax.set_xlabel("시간(시)"); ax.set_ylabel("하루 평균 승차 (연결된 역 합계)"); ax.yaxis.set_major_formatter(_thousands)
    ax.legend(frameon=False, labelcolor=INK2)
    _title(ax, "일별 추정 지하철 승차: 요일구분별 시간 패턴", f"물리역 {df_subway_full['phys_key'].nunique()}곳 · 평일 출퇴근 피크가 보이면 정상")
    fig.tight_layout(); plt.show()
else:
    print("subway_ride_day_hour_full 없음 (01 노트북 6절에서 분석 기간 실측이 없었음) → 생략")